# GradeSense — Week 1: Data Exploration & TF-IDF Baseline

**Goal of this notebook**
1. Load SciEntsBank and understand its structure
2. Explore label distribution and answer lengths
3. Build Baseline 1: TF-IDF features + Logistic Regression
4. Score it on all three test splits using the shared `src/evaluate.py`

Splits: `train` → we carve out `val` (10%) · `test_ua` unseen answers · `test_uq` unseen questions · `test_ud` unseen domains

In [ ]:
import sys, os
# Find the project root (the folder containing src/), whatever folder the notebook runs from
ROOT = os.path.abspath(".")
while not os.path.isdir(os.path.join(ROOT, "src")) and os.path.dirname(ROOT) != ROOT:
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_splits, label_names
from src.evaluate import evaluate_all_splits, compute_metrics, RESULTS_DIR
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Project root:", ROOT)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

## 1. Load the data

In [ ]:
SCHEME = "5way"            # try "3way" and "2way" later
splits = load_splits(scheme=SCHEME)
LABELS = label_names(SCHEME)
train = splits["train"]

for name, df in splits.items():
    print(f"{name:8s}: {len(df):5d} rows | {df['question'].nunique():3d} unique questions")

In [ ]:
splits["train"][["question", "reference_answer", "student_answer", "label_name"]].head(8)

## 2. Label distribution

Classes are imbalanced (`non_domain` is tiny). That's why we report **macro-F1**,
not just accuracy — a model can score high accuracy while ignoring rare classes.

In [ ]:
dist = pd.DataFrame({
    name: df["label_name"].value_counts(normalize=True).reindex(LABELS).fillna(0)
    for name, df in splits.items()
})
display((dist * 100).round(1).astype(str) + "%")

ax = dist.T.plot(kind="bar", stacked=True, figsize=(9, 4), colormap="tab10")
ax.set_ylabel("Proportion"); ax.set_title("Label distribution per split")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left"); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "label_distribution.png"), dpi=150); plt.show()

## 3. Answer lengths

This decides `max_len` for the transformer next week. We check the
**combined** length of question + reference + student answer in tokens.
(First run downloads the BERT tokenizer, ~1 MB.)

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("bert-base-uncased")

pair_lens = [
    len(tok(q + " " + r, s)["input_ids"])
    for q, r, s in zip(train["question"], train["reference_answer"], train["student_answer"])
]
train_words = train["student_answer"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(train_words, bins=40); axes[0].set_title("Student answer length (words)")
axes[1].hist(pair_lens, bins=40);   axes[1].set_title("Full input length (BERT tokens)")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "length_distribution.png"), dpi=150); plt.show()

print("Student answer words  — mean %.1f, 95th pct %d" % (train_words.mean(), np.percentile(train_words, 95)))
print("Full input tokens     — mean %.1f, 95th pct %d, max %d" % (np.mean(pair_lens), np.percentile(pair_lens, 95), max(pair_lens)))

## 4. Baseline 1 — TF-IDF + Logistic Regression

Two versions:

- **TFIDF-Sim**: only *similarity* features (does the answer resemble the reference?):
  cosine similarity to reference and question, word overlap, length ratio, answer length.
- **TFIDF-Full**: similarity features **plus** the TF-IDF vector of the student answer itself.

Prediction to check: *Full* should do better on unseen answers (it memorises question-specific
words) but worse on unseen questions/domains. That gap is a key viva point.

In [ ]:
import re
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def clean(t):
    return re.sub(r"[^a-z0-9 ]", " ", str(t).lower())

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, stop_words="english", sublinear_tf=True)
vec.fit(pd.concat([train["reference_answer"], train["student_answer"], train["question"]]).map(clean))

def sim_features(df):
    R = vec.transform(df["reference_answer"].map(clean))
    S = vec.transform(df["student_answer"].map(clean))
    Q = vec.transform(df["question"].map(clean))
    cos_rs = np.asarray(R.multiply(S).sum(axis=1)).ravel()   # rows are L2-normalised → dot = cosine
    cos_qs = np.asarray(Q.multiply(S).sum(axis=1)).ravel()
    feats = []
    for r, s in zip(df["reference_answer"].map(clean), df["student_answer"].map(clean)):
        rw, sw = set(r.split()), set(s.split())
        overlap = len(rw & sw) / max(len(rw), 1)
        len_ratio = len(sw) / max(len(rw), 1)
        feats.append([overlap, len_ratio, len(sw)])
    return np.column_stack([cos_rs, cos_qs, np.array(feats)])

scaler = StandardScaler().fit(sim_features(train))

def X_sim(df):
    return scaler.transform(sim_features(df))

def X_full(df):
    return hstack([csr_matrix(X_sim(df)), vec.transform(df["student_answer"].map(clean))]).tocsr()

In [ ]:
clf_sim = LogisticRegression(max_iter=2000, class_weight="balanced")
clf_sim.fit(X_sim(train), train["label"])

res_sim = evaluate_all_splits(
    "TFIDF-Sim", lambda df: clf_sim.predict(X_sim(df)),
    splits, LABELS, SCHEME,
)

In [ ]:
clf_full = LogisticRegression(max_iter=3000, class_weight="balanced", C=2.0)
clf_full.fit(X_full(train), train["label"])

res_full = evaluate_all_splits(
    "TFIDF-Full", lambda df: clf_full.predict(X_full(df)),
    splits, LABELS, SCHEME,
)

## 5. Compare & save

In [ ]:
results = pd.read_csv(os.path.join(RESULTS_DIR, "results.csv"))
pivot = results[results["scheme"] == SCHEME].pivot(index="model", columns="split", values="macro_f1")
pivot = pivot[[c for c in ["val", "test_ua", "test_uq", "test_ud"] if c in pivot.columns]]
display(pivot.round(3))

pivot.drop(columns=["val"], errors="ignore").T.plot(kind="bar", figsize=(7, 3.5))
plt.ylabel("Macro-F1"); plt.title(f"Baselines — {SCHEME}"); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, f"baselines_{SCHEME}.png"), dpi=150); plt.show()

## 6. Error analysis — where does the baseline fail?

Paraphrased correct answers with few shared words are the classic TF-IDF failure.
Copy a few examples into your report notes; next week we check whether DeBERTa fixes them.

In [ ]:
ua = splits["test_ua"].copy()
ua["pred"] = [LABELS[i] for i in clf_sim.predict(X_sim(ua))]
wrong = ua[ua["pred"] != ua["label_name"]]
print(f"{len(wrong)} / {len(ua)} wrong on test_ua")
wrong[["reference_answer", "student_answer", "label_name", "pred"]].sample(min(10, len(wrong)), random_state=1)

## Week 1 checklist
- [ ] Label distribution + length plots saved in `results/`
- [ ] Note the 95th-percentile token length → choose `max_len` (likely 128–256)
- [ ] Baseline scores in `results/results.csv`
- [ ] 5–10 failure examples copied into report notes
- [ ] Optional: set `SCHEME = "3way"` / `"2way"` and re-run all cells